# Ray API Tour

A short, didactic walk through Ray's four core APIs in isolation:

- **Ray Core** — `@ray.remote` for task parallelism
- **Ray Data** — `ray.data.from_pandas` for distributed datasets
- **Ray Tune** — `tune.run` for hyperparameter search
- **Ray Serve** — `@serve.deployment` for model serving

Each section uses the smallest possible self-contained example so you
can read it as documentation rather than as an applied project. For
the same APIs used end-to-end on a real regression problem, see
`ray_housing.example.ipynb`.

> Run the cells top to bottom. Some sections call `ray.init()` or
> `serve.run(...)` which can take a few seconds the first time.

## 1. Ray Core: `@ray.remote`

`@ray.remote` is the most basic Ray primitive. It turns an ordinary
function into a "remote function" that returns a `ObjectRef` future
instead of a value. Ray schedules invocations across all available
CPU cores; you collect the results with `ray.get(...)`.

The decorator is the *only* code change needed to parallelize a
function across cores or — with no further change — across a Ray
cluster.

In [1]:
import ray

# ray.init starts a single-node Ray runtime in this process.
# ignore_reinit_error makes re-running the cell harmless.
ray.init(ignore_reinit_error=True)

2026-05-05 21:25:48,502	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-05-05 21:25:54,066	WARNING services.py:2148 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 67108864 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=1.05gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.
2026-05-05 21:25:54,188	INFO worker.py:1942 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8266 


Python version:,3.12.13
Ray version:,2.49.0
Dashboard:,http://127.0.0.1:8266


In [2]:
@ray.remote
def square(x):
    """A trivial remote function for demonstration."""
    return x * x

# .remote(...) submits the call and returns a future.
# ray.get([...]) waits for all futures and returns the values.
futures = [square.remote(i) for i in range(8)]
print("Results:", ray.get(futures))

Results: [0, 1, 4, 9, 16, 25, 36, 49]


## 2. Ray Data: `ray.data.from_pandas`

`ray.data` is Ray's distributed dataset library. Datasets are lazy,
parallelized, and can be processed with familiar transformations like
`map_batches`, `filter`, and `groupby`. The smallest possible example
is wrapping a pandas DataFrame.

In [3]:
import pandas as pd
import ray

# A toy DataFrame with five rows.
df = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "value": [10, 20, 30, 40, 50],
})

# Wrap it in a Ray Dataset.
ds = ray.data.from_pandas(df)
ds.show()

2026-05-05 21:26:00,271	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-05-05 21:26:01,245	INFO dataset.py:3246 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-05-05 21:26:01,254	INFO logging.py:295 -- Registered dataset logger for dataset dataset_1_0
2026-05-05 21:26:01,268	INFO streaming_executor.py:159 -- Starting execution of Dataset dataset_1_0. Full logs are in /tmp/ray/session_2026-05-05_21-25-48_524751_9711/logs/ray-data
2026-05-05 21:26:01,269	INFO streaming_executor.py:160 -- Execution plan of Dataset dataset_1_0: InputDataBuffer[Input] -> LimitOperator[limit=20]
2026-05-05 21:26:01,271	WARNING resource_manager.py:134 -- ⚠️  Ray's object store is configured to use only 42.9% of available memory (1.0GiB out of 2.2GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% o

[dataset]: Run `pip install tqdm` to enable progress reporting.
{'id': 1, 'value': 10}
{'id': 2, 'value': 20}
{'id': 3, 'value': 30}
{'id': 4, 'value': 40}
{'id': 5, 'value': 50}


In [4]:
# Datasets support batched transformations. map_batches applies a
# function to chunks of rows in parallel. Here we double "value".
def double_value(batch):
    batch["value"] = batch["value"] * 2
    return batch

ds_doubled = ds.map_batches(double_value)
ds_doubled.show()

2026-05-05 21:26:01,327	INFO logging.py:295 -- Registered dataset logger for dataset dataset_3_0
2026-05-05 21:26:01,330	INFO streaming_executor.py:159 -- Starting execution of Dataset dataset_3_0. Full logs are in /tmp/ray/session_2026-05-05_21-25-48_524751_9711/logs/ray-data
2026-05-05 21:26:01,331	INFO streaming_executor.py:160 -- Execution plan of Dataset dataset_3_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(double_value)] -> LimitOperator[limit=20]
2026-05-05 21:26:01,374	INFO streaming_executor.py:279 -- ✔️  Dataset dataset_3_0 execution finished in 0.04 seconds


{'id': 1, 'value': 20}
{'id': 2, 'value': 40}
{'id': 3, 'value': 60}
{'id': 4, 'value': 80}
{'id': 5, 'value': 100}


## 3. Ray Tune: `tune.run`

Ray Tune is a hyperparameter search library built on top of Ray Core.
You define a "trainable" — a function that takes a `config` dict and
calls `tune.report({...})` with a metric — and Tune handles the
parallel scheduling and result aggregation.

The example below sweeps a single parameter `x` over a small grid and
finds the value that minimizes `(x - 3)**2`.

In [5]:
from ray import tune

def trainable(config):
    """A trivial trainable: parabola minimized at x=3."""
    score = (config["x"] - 3) ** 2
    tune.report({"score": score})

analysis = tune.run(
    trainable,
    config={"x": tune.grid_search([1, 2, 3, 4, 5])},
    metric="score",
    mode="min",
    verbose=0,  # quiet output for tutorial readability
)

print("Best config:", analysis.get_best_config(metric="score", mode="min"))

2026-05-05 21:26:07,309	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/root/ray_results/trainable_2026-05-05_21-26-01' in 0.0055s.


Best config: {'x': 3}


## 4. Ray Serve: `@serve.deployment`

Ray Serve lets you deploy a Python class as an HTTP endpoint with a
single decorator. It runs inside the same Ray runtime — no separate
server process — and scales horizontally just by changing a replica
count.

The minimal example below exposes a `/` endpoint that returns a JSON
greeting.

In [12]:
from ray import serve
from starlette.responses import JSONResponse

@serve.deployment
class Hello:
    """A trivial Serve deployment that returns a static greeting."""

    async def __call__(self, request):
        # Returning a JSONResponse explicitly avoids any auto-conversion
        # quirks in Ray Serve's starlette integration.
        return JSONResponse({"message": "hello from Ray Serve"})


# serve.run starts Serve if it isn't already running, registers the
# deployment, and binds the HTTP proxy on port 8000.
serve.run(Hello.bind())

INFO 2026-05-05 21:29:38,110 serve 9711 -- Connecting to existing Serve app in namespace "serve". New http options will not be applied.
(ServeController pid=10423) INFO 2026-05-05 21:29:38,200 controller 10423 -- Deploying new version of Deployment(name='Hello', app='default') (initial target replicas: 1).
(ServeController pid=10423) INFO 2026-05-05 21:29:38,306 controller 10423 -- Stopping 1 replicas of Deployment(name='Hello', app='default') with outdated versions.
(ServeController pid=10423) INFO 2026-05-05 21:29:38,306 controller 10423 -- Adding 1 replica to Deployment(name='Hello', app='default').
(ServeController pid=10423) INFO 2026-05-05 21:29:40,349 controller 10423 -- Replica(id='h6fky95n', deployment='Hello', app='default') is stopped.
INFO 2026-05-05 21:29:41,129 serve 9711 -- Application 'default' is ready at http://127.0.0.1:8000/.


DeploymentHandle(deployment='Hello')

In [13]:
import requests

response = requests.get("http://127.0.0.1:8000/")
print("Status:", response.status_code)
print("Body:  ", response.json())

Status: 200
Body:   {'message': 'hello from Ray Serve'}


(ServeReplica:default:Hello pid=11929) INFO 2026-05-05 21:29:49,977 default_Hello uyqmoeyz d3b7e737-4afc-46c3-aa85-1b9cb691a331 -- GET / 200 1.7ms


## Wrapping Up

That's the end of the API tour. With those four primitives —
`@ray.remote`, `ray.data.from_pandas`, `tune.run`, and
`@serve.deployment` — you have everything Ray uses to scale
ML workloads from a laptop to a multi-node cluster.

Now head over to `ray_housing.example.ipynb` to see the same APIs
applied to a real regression problem.